# PROMISE Multi-Project CFG Extraction

This notebook runs and inspects the generalized Soot-based CFG extraction pipeline implemented in `scripts/extract_promise_cfg.py`.

The script is the single source of truth. The notebook only wraps execution and validates the generated graph/tensor outputs.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import numpy as np
import pandas as pd

REPO_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
SCRIPT_PATH = REPO_ROOT / 'scripts' / 'extract_promise_cfg.py'
INPUT_CSV = REPO_ROOT / 'outputs' / 'promise' / 'promise_preprocessed_standard.csv'
CFG_DIR = REPO_ROOT / 'outputs' / 'promise' / 'cfg'
SUMMARY_JSON = CFG_DIR / 'cfg_summary.json'
GRAPH_INDEX = CFG_DIR / 'graph_index.csv'
REPORT_MD = CFG_DIR / 'cfg_report.md'
VALIDATION_JSON = CFG_DIR / 'validation_issues.json'

print('Repository:', REPO_ROOT)
print('CFG extractor:', SCRIPT_PATH)
print('Input CSV:', INPUT_CSV)
print('CFG output directory:', CFG_DIR)

## 1. Design

The extractor creates one file-level CFG graph for every mapped Java class in the combined preprocessed PROMISE dataset.

Each file-level graph aggregates all concrete method/constructor CFGs recovered by Soot:

- one synthetic `ENTRY` and `EXIT` node per method
- statement-level CFG nodes from Soot Jimple units
- typed control-flow edges: `CFG_NEXT`, `CFG_TRUE`, `CFG_FALSE`, `CFG_RETURN`, `CFG_EXCEPTION`

The GNN tensor contract matches the AST pipeline style:

- structural node features: `x = [normalized_line_position, has_source_line, is_synthetic]`
- exact CFG node type ids: `node_type_id`
- directed control-flow edges: `edge_index`
- edge relation ids: `edge_type`

During GNN training, `node_type_id` should be passed through a trainable embedding layer and concatenated with `x`. The `edge_type` tensor should be used by a relation-aware GNN layer.

In [ ]:
pre_df = pd.read_csv(INPUT_CSV)
print('combined preprocessing shape:', pre_df.shape)
print('datasets:', sorted(pre_df['dataset_name'].unique()))
print('mapped rows:', int(pre_df['source_path'].notna().sum()))
pre_df.head()

## 2. Run CFG Extraction

Terminal equivalent:

```bash
python scripts/extract_promise_cfg.py
```

To run one dataset only:

```bash
python scripts/extract_promise_cfg.py --dataset-name ant-1.6
```

The script compiles each source project with ECJ, runs the Soot helper, and writes logs under `outputs/promise/cfg/logs/`.

In [ ]:
result = subprocess.run(
    [sys.executable, str(SCRIPT_PATH)],
    cwd=REPO_ROOT,
    text=True,
    capture_output=True,
    check=True,
)
print(result.stdout)
if result.stderr:
    print(result.stderr)

## 3. Extraction Summary

In [ ]:
summary = json.loads(SUMMARY_JSON.read_text())
summary_view = {
    key: summary[key]
    for key in [
        'input_rows',
        'requested_mapped_files',
        'graphs_generated',
        'soot_graphs',
        'placeholder_graphs',
        'parse_failures',
        'validation_issues',
        'total_methods',
        'total_nodes',
        'total_edges',
        'avg_nodes_per_file',
        'avg_edges_per_file',
        'structural_feature_dim',
        'node_type_embedding_dim',
        'model_node_feature_dim',
        'node_type_vocab_size',
        'edge_type_vocab_size',
        'backend',
    ]
}
summary_view

## 4. Per-Dataset Results

In [ ]:
pd.DataFrame(summary['datasets'])

## 5. Graph Index

In [ ]:
graph_index = pd.read_csv(GRAPH_INDEX)
print('graph_index shape:', graph_index.shape)
graph_index.head()

## 6. Inspect One CFG Graph and Tensor Set

In [ ]:
example = graph_index[graph_index['extraction_mode'] == 'soot'].iloc[0] if (graph_index['extraction_mode'] == 'soot').any() else graph_index.iloc[0]
print(example[['graph_id', 'dataset_name', 'name', 'extraction_mode', 'num_nodes', 'num_edges', 'num_methods']])

graph = json.loads(Path(example['graph_json']).read_text())
x = np.load(example['x_npy'])
node_type_id = np.load(example['node_type_id_npy'])
edge_index = np.load(example['edge_index_npy'])
edge_type = np.load(example['edge_type_npy'])

print('x shape:', x.shape)
print('node_type_id shape:', node_type_id.shape)
print('edge_index shape:', edge_index.shape)
print('edge_type shape:', edge_type.shape)
print('first nodes:')
pd.DataFrame(graph['nodes']).head(10)

In [ ]:
print('first edges:')
pd.DataFrame(graph['edges']).head(10)

## 7. Validation Checks

In [ ]:
validation_issues = json.loads(VALIDATION_JSON.read_text())
checks = {
    'graphs_match_requested': summary['graphs_generated'] == summary['requested_mapped_files'],
    'graph_index_rows_match_summary': len(graph_index) == summary['graphs_generated'],
    'all_x_feature_dim_is_3': graph_index['node_feature_dim'].eq(3).all(),
    'model_feature_dim_is_35': summary['model_node_feature_dim'] == 35,
    'validation_issues_count_matches': len(validation_issues) == summary['validation_issues'],
}
checks

## 8. Report Preview

In [ ]:
report_text = REPORT_MD.read_text()
print('
'.join(report_text.splitlines()[:80]))